# Feature Engineering, Transformation & Class Imbalance

## Objective
Prepare clean, feature-rich datasets ready for modeling:
- **Fraud_Data.csv**: geolocation merge, feature engineering, encoding, scaling, SMOTE
- **creditcard.csv**: scaling, SMOTE

**Resampling (SMOTE) applied only on the training set** to avoid data leakage.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
print('Libraries loaded')

ModuleNotFoundError: No module named 'imblearn'

---
## Part A: Fraud_Data.csv

### A1. Load, Clean, Geolocation Merge

In [ ]:
fraud = pd.read_csv('../data/raw/Fraud_Data.csv', parse_dates=['signup_time','purchase_time'])
ip_map = pd.read_csv('../data/raw/IpAddress_to_Country.csv')

fraud = fraud.drop_duplicates().copy()
fraud['ip_address_int'] = fraud['ip_address'].round().astype('int64')
fraud['class'] = fraud['class'].astype('int8')

fs = fraud.sort_values('ip_address_int').reset_index(drop=True)
ips = ip_map.sort_values('lower_bound_ip_address').reset_index(drop=True)
fg = pd.merge_asof(fs, ips[['lower_bound_ip_address','upper_bound_ip_address','country']],
    left_on='ip_address_int', right_on='lower_bound_ip_address', direction='backward')
mask = fg['ip_address_int'] <= fg['upper_bound_ip_address']
fg.loc[~mask | fg['country'].isna(), 'country'] = 'Unknown'
print(f'After merge: {len(fg):,} rows, country matched: {(fg["country"]!="Unknown").sum():,}')

### A2. Feature Engineering

In [ ]:
fg['time_since_signup'] = (
    (fg['purchase_time']-fg['signup_time']).dt.total_seconds()/3600).clip(lower=0)
fg['hour_of_day'] = fg['purchase_time'].dt.hour
fg['day_of_week'] = fg['purchase_time'].dt.dayofweek

uc = fg.groupby('user_id').size().reset_index(name='user_txn_count')
fg = fg.merge(uc, on='user_id')

fg = fg.sort_values(['user_id','purchase_time']).reset_index(drop=True)
def count_24h(s):
    return s.apply(lambda t: ((s>=t-pd.Timedelta(hours=24))&(s<=t)).sum())
fg['txn_in_24h'] = fg.groupby('user_id')['purchase_time'].transform(count_24h)
print('Features created.')
print(fg[['time_since_signup','hour_of_day','day_of_week','user_txn_count','txn_in_24h']].describe())

### A3. Feature Matrix & Train-Test Split

In [ ]:
num_features = ['purchase_value','age','time_since_signup',
                'hour_of_day','day_of_week','user_txn_count','txn_in_24h']
cat_features = ['source','browser','sex','country']

X = fg[num_features + cat_features].copy()
y = fg['class'].copy()
print(f'Features: {X.shape}, Target:\n{y.value_counts()}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')

### A4. Encode & Scale

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
])

X_train_t = pd.DataFrame(preprocessor.fit_transform(X_train), index=X_train.index)
X_test_t  = pd.DataFrame(preprocessor.transform(X_test), index=X_test.index)

ohe = preprocessor.named_transformers_['cat']
cat_names = ohe.get_feature_names_out(cat_features).tolist()
X_train_t.columns = num_features + cat_names
X_test_t.columns  = num_features + cat_names
print(f'Transformed: train {X_train_t.shape}, test {X_test_t.shape}')
print(f'Features ({len(X_train_t.columns)}): {list(X_train_t.columns)}')

### A5. SMOTE (Training Set Only)

**Why SMOTE?**
1. Fraud class is ~9% - severe but not extreme imbalance
2. SMOTE generates synthetic samples along the feature-space convex hull, preserving decision boundaries
3. Applied **only on training data** to prevent data leakage

In [ ]:
print('BEFORE SMOTE:')
print(y_train.value_counts())
print(f'Fraud ratio: {y_train.mean()*100:.2f}%')

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_t, y_train)

print('\nAFTER SMOTE:')
print(pd.Series(y_train_res).value_counts())
print(f'Fraud ratio: {pd.Series(y_train_res).mean()*100:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
y_train.value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','coral'], edgecolor='black')
axes[0].set_title('Before SMOTE', fontweight='bold')
axes[0].set_xticklabels(['Legit','Fraud'], rotation=0)
pd.Series(y_train_res).value_counts().plot(kind='bar', ax=axes[1], color=['steelblue','coral'], edgecolor='black')
axes[1].set_title('After SMOTE', fontweight='bold')
axes[1].set_xticklabels(['Legit','Fraud'], rotation=0)
plt.tight_layout()
plt.savefig('../data/processed/fraud_smote_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### A6. Save Fraud_Data Splits

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
X_train_res.to_csv('../data/processed/fraud_X_train_smote.csv', index=False)
pd.Series(y_train_res).to_csv('../data/processed/fraud_y_train_smote.csv', index=False, header=['class'])
X_test_t.to_csv('../data/processed/fraud_X_test.csv', index=False)
y_test.to_csv('../data/processed/fraud_y_test.csv', index=False, header=['class'])
print('Saved fraud_X_train_smote.csv, fraud_y_train_smote.csv, fraud_X_test.csv, fraud_y_test.csv')

---
## Part B: creditcard.csv

### B1. Load, Clean, Split, Scale, SMOTE

In [ ]:
cc = pd.read_csv('../data/raw/creditcard.csv')
cc = cc.drop_duplicates().copy()
cc['Class'] = cc['Class'].astype('int8')
print(f'Rows: {len(cc):,}, Fraud ratio: {cc["Class"].mean()*100:.4f}%')

X_cc = cc.drop(columns=['Class'])
y_cc = cc['Class']
Xtr, Xte, ytr, yte = train_test_split(X_cc, y_cc, test_size=0.2, random_state=42, stratify=y_cc)
print(f'Train: {len(Xtr):,}  Test: {len(Xte):,}')

sc = StandardScaler()
Xtr_s = pd.DataFrame(sc.fit_transform(Xtr), columns=Xtr.columns, index=Xtr.index)
Xte_s = pd.DataFrame(sc.transform(Xte), columns=Xte.columns, index=Xte.index)

print('BEFORE SMOTE:', ytr.value_counts().to_dict())
sm2 = SMOTE(random_state=42)
Xtr_res, ytr_res = sm2.fit_resample(Xtr_s, ytr)
print('AFTER SMOTE:', pd.Series(ytr_res).value_counts().to_dict())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ytr.value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','coral'], edgecolor='black')
axes[0].set_title('CC Before SMOTE', fontweight='bold'); axes[0].set_yscale('log')
axes[0].set_xticklabels(['Legit','Fraud'], rotation=0)
pd.Series(ytr_res).value_counts().plot(kind='bar', ax=axes[1], color=['steelblue','coral'], edgecolor='black')
axes[1].set_title('CC After SMOTE', fontweight='bold')
axes[1].set_xticklabels(['Legit','Fraud'], rotation=0)
plt.tight_layout()
plt.savefig('../data/processed/creditcard_smote_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### B2. Save creditcard Splits

In [ ]:
Xtr_res_df = pd.DataFrame(Xtr_res, columns=Xtr.columns)
Xtr_res_df.to_csv('../data/processed/cc_X_train_smote.csv', index=False)
pd.Series(ytr_res).to_csv('../data/processed/cc_y_train_smote.csv', index=False, header=['Class'])
Xte_s.to_csv('../data/processed/cc_X_test.csv', index=False)
yte.to_csv('../data/processed/cc_y_test.csv', index=False, header=['Class'])
print('Saved cc_X_train_smote.csv, cc_y_train_smote.csv, cc_X_test.csv, cc_y_test.csv')

## Summary

| Dataset | Rows (clean) | Train (SMOTE) | Test | Fraud Rate (before/after) |
|---|---|---|---|---|
| Fraud_Data | ~151K | balanced | 20% holdout | ~9% -> 50% |
| creditcard | ~285K | balanced | 20% holdout | ~0.17% -> 50% |

**Key decisions:**
1. SMOTE on training set only - prevents data leakage
2. StandardScaler for all numerical features
3. One-hot encoding for Fraud_Data categoricals
4. Both datasets saved as train/test splits ready for modeling